[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alokothro/movie-recommender-svd/blob/main/notebooks/04_what_it_learned.ipynb)

# Milestone 4: What Did It Learn, and What Would It Recommend to Me?
*MATH 344L Project 2: A Movie Recommendation System*

This notebook names the hidden taste factors the SVD found in MovieLens, then adds me as a new user with 14 ratings to see how the model's personal part changes my recommendations.

In Milestone 3 the SVD model beat the baseline (test RMSE 0.9286 vs 0.9545). But a number like that doesn't say *what* the model actually learned. In this notebook I open the model up and do two things:

1. Look at the **hidden factors** the SVD found and try to give them names.
2. Add **myself** as a new user and see what it recommends to me.

At the end there's one finding, in plain words, that also goes in the README.

## 1. Set up

In [1]:
import os
import subprocess
import sys

USER = "Alokothro"
REPO = "movie-recommender-svd"

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", f"https://github.com/{USER}/{REPO}.git", REPO],
        check=True)

sys.path.insert(0, f"{REPO}/src")
from evaluate import to_matrix
from models import baseline_matrix, rank_k

Cloning into 'movie-recommender-svd'...


This is the same first cell as Milestone 3. It clones my repo from GitHub (or just updates it with `git pull` if it's already there) and adds the `src` folder to the places Python looks for code. This time I only need three functions: `to_matrix` from `evaluate.py`, plus `baseline_matrix` and `rank_k` from `models.py`, which I saved there in Milestone 3. Any message under the cell is just git saying what it did.

In [2]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])
print("Number of ratings:", len(ratings))

Number of ratings: 100000


This is the same download cell as Milestone 3. It only downloads MovieLens if the `ml-100k` folder isn't there yet, then reads all 100,000 ratings into a table. This time there's no train/test split. Section 2 explains why.

In [3]:
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

1682 Star Wars (1977)


This reads the movie titles from `u.item`. Each line of that file is one movie, with the pieces separated by `|`, and column 1 is the title. `usecols=[1]` keeps just that column, and `[1].to_numpy()` turns it into a plain array of 1,682 titles. `encoding="latin-1"` is needed because some titles have accented letters. Movie number 50 is `titles[49]` because Python counts from 0, and it's *Star Wars (1977)*.

## 2. Fit Model 2 on all the ratings

In [4]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)

n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


This is Model 2 from Milestone 3 (the baseline plus an SVD of the deviations), with one change: it uses `ratings`, *all* 100,000 of them, instead of just `train`. In Milestone 3 we were *measuring* the model, so we had to hide a test set. Now we're *using* it, so we give it everything we have.

- `B` is the baseline prediction for every user and movie, and `X` is how far each known rating is from the baseline, with 0 in the gaps.
- `np.linalg.svd(X, ...)` splits `X` into U, the singular values `s`, and Vᵀ. Each row of `Vt` is one hidden factor, with one number for every movie.
- `n_ratings` counts how many ratings each movie has, and `popular` is True for the 603 movies with at least 50. We only look at those, because a movie rated by 3 people can land anywhere just by chance.

## 3. What did it learn? Naming the hidden factors

In [5]:
def show_factor(f, n=6):
    """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
    v = Vt[f - 1]
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]
    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")
    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")

show_factor(1)

Factor 1, one end:
  -0.087 Clockwork Orange, A (1971)
  -0.079 Trainspotting (1996)
  -0.072 Leaving Las Vegas (1995)
  -0.072 Pulp Fiction (1994)
  -0.069 Brazil (1985)
  -0.067 Fargo (1996)
Factor 1, other end:
  +0.211 Independence Day (ID4) (1996)
  +0.175 Return of the Jedi (1983)
  +0.149 Raiders of the Lost Ark (1981)
  +0.149 Back to the Future (1985)
  +0.136 Braveheart (1995)
  +0.134 Forrest Gump (1994)


`show_factor` prints the popular movies at the two ends of one hidden factor.

- `v = Vt[f - 1]` is that factor's row: one score for every movie (factor 1 is row 0).
- `np.where(popular)[0]` is the list of the popular movies' positions, and `np.argsort` sorts them by their score, from lowest to highest.
- `order[:n]` is the first `n` (the lowest scores). `order[::-1]` flips the list, so `order[::-1][:n]` is the highest `n`.

**Factor 1:** one end is dark, weird, R-rated movies (*A Clockwork Orange*, *Trainspotting*, *Leaving Las Vegas*, *Pulp Fiction*, *Brazil*, *Fargo*). The other end is big, feel-good crowd-pleasers (*Independence Day*, *Return of the Jedi*, *Raiders of the Lost Ark*, *Back to the Future*, *Braveheart*, *Forrest Gump*). Nobody told the SVD about genres or age ratings. It found this split only from who liked what.

**The signs might be flipped.** An SVD can swap uᵢ and vᵢ for −uᵢ and −vᵢ without changing anything, so on a different computer (like Colab) the two ends might trade places and the numbers might have the opposite sign. The movies at each end should still be the same.

In [6]:
show_factor(3)

Factor 3, one end:
  -0.137 Independence Day (ID4) (1996)
  -0.131 Star Trek: First Contact (1996)
  -0.130 Rock, The (1996)
  -0.129 Alien (1979)
  -0.121 Monty Python and the Holy Grail (1974)
  -0.119 Aliens (1986)
Factor 3, other end:
  +0.163 Sound of Music, The (1965)
  +0.144 Grease (1978)
  +0.099 E.T. the Extra-Terrestrial (1982)
  +0.087 Mary Poppins (1964)
  +0.085 Jaws 2 (1978)
  +0.084 Snow White and the Seven Dwarfs (1937)


**Factor 3:** one end is mostly sci-fi and action (*Independence Day*, *Star Trek: First Contact*, *The Rock*, *Alien*, *Aliens*), plus *Monty Python and the Holy Grail*, which doesn't really fit. The other end is musicals and family movies (*The Sound of Music*, *Grease*, *E.T.*, *Mary Poppins*, *Snow White and the Seven Dwarfs*), plus *Jaws 2*, which also doesn't really fit. So it's mostly "sci-fi and action vs. musicals and family movies," but it isn't perfectly clean. Not every factor is even this clear, which Exercise 1 gets into.

## 4. What would it recommend to me?
### Step 1: find movie numbers

In [7]:
def find(word):
    """Print the ID and title of every movie whose title contains word."""
    for m, t in enumerate(titles):
        if word.lower() in t.lower():
            print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


`find` searches every title for a word. `enumerate(titles)` gives each title together with its position `m`, and `.lower()` makes both the word and the title lowercase, so capital letters don't matter. It prints `m + 1` because MovieLens IDs start at 1. Searching "alien" finds all four *Alien* movies, and *Alien (1979)* is movie 183.

In [8]:
for word in ["toy story", "lion king", "pulp fiction", "shawshank", "godfather",
             "back to the future", "jurassic park", "men in black", "home alone",
             "titanic", "independence day", "twister", "grease", "dirty dancing"]:
    find(word)

1 Toy Story (1995)
71 Lion King, The (1994)
56 Pulp Fiction (1994)
64 Shawshank Redemption, The (1994)
127 Godfather, The (1972)
187 Godfather: Part II, The (1974)
204 Back to the Future (1985)
82 Jurassic Park (1993)
252 Lost World: Jurassic Park, The (1997)
257 Men in Black (1997)
94 Home Alone (1990)
894 Home Alone 3 (1997)
313 Titanic (1997)
121 Independence Day (ID4) (1996)
118 Twister (1996)
451 Grease (1978)
1037 Grease 2 (1982)
155 Dirty Dancing (1987)


I used `find` to look up the ID of every movie I wanted to rate. Some searches match more than one movie (like *The Godfather: Part II*, *The Lost World: Jurassic Park*, *Home Alone 3*, or *Grease 2*), so I picked the original movie out of each list.

### Step 2: rate 10 to 15 movies I've seen

In [9]:
my_ratings = {
    1: 5,      # Toy Story
    71: 4,     # The Lion King
    56: 5,     # Pulp Fiction
    64: 5,     # The Shawshank Redemption
    127: 4,    # The Godfather
    204: 4,    # Back to the Future
    82: 4,     # Jurassic Park
    257: 3,    # Men in Black
    94: 3,     # Home Alone
    313: 2,    # Titanic
    121: 2,    # Independence Day
    118: 1,    # Twister
    451: 1,    # Grease
    155: 1,    # Dirty Dancing
}
print(len(my_ratings), "ratings, average", round(np.mean(list(my_ratings.values())), 2))

14 ratings, average 3.14


These are my ratings. A dictionary `{movie: stars, ...}` pairs each movie ID with how many stars I gave it, and the comments are just there so I can remember which movie is which. The data is from 1998, so these are all older movies.

My taste in short: I love Pixar and the big acclaimed crime and drama movies (*Toy Story*, *Pulp Fiction*, and *The Shawshank Redemption* get 5 stars). I'm in the middle on family comedies (*Men in Black* and *Home Alone* get 3). And I don't like the 1996 disaster blockbusters or musicals and romance (*Twister*, *Grease*, and *Dirty Dancing* get 1). My average is 3.14 stars, which is lower than the overall MovieLens average of about 3.53.

### Step 3: add myself as user 944 and refit

In [10]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


`me` is a one-row matrix that's all `nan` except for my ratings. The loop puts each rating in column `movie - 1` (minus 1 again because Python counts from 0). `np.vstack` stacks `me` under `R` as a new last row, so the matrix goes from 943 users to 944, and I'm user 944.

The rest is Model 2 again on the bigger matrix: recompute the baseline (so now it knows how generous a rater I am), take the deviations, do the SVD, and keep k = 10 factors. The assignment uses k = 10. In my Milestone 3, k = 15 was technically the best for Model 2, but k = 10 was almost exactly as good (0.9511 vs 0.9492 validation RMSE), so 10 is a fine choice. My predictions are the last row, `P[-1]`.

### Step 4: compare with the baseline

In [11]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    """The n highest-scoring movies among those we may recommend."""
    return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
    print(f"  {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
    print(f"  {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
  4.08 Close Shave, A (1995)
  4.00 Wrong Trousers, The (1993)
  3.98 Usual Suspects, The (1995)
  3.97 Star Wars (1977)
  3.96 Casablanca (1942)
  3.95 Schindler's List (1993)
  3.93 Wallace & Gromit: The Best of Aardman Animation (1996)
  3.89 Good Will Hunting (1997)
  3.87 Rear Window (1954)
  3.85 12 Angry Men (1957)
Baseline recommends:
  4.08 Close Shave, A (1995)
  3.99 Wrong Trousers, The (1993)
  3.97 Schindler's List (1993)
  3.94 Casablanca (1942)
  3.92 Wallace & Gromit: The Best of Aardman Animation (1996)
  3.92 Usual Suspects, The (1995)
  3.90 Good Will Hunting (1997)
  3.89 Star Wars (1977)
  3.85 12 Angry Men (1957)
  3.84 Rear Window (1954)
in both lists: 10


- `mine` is my row of Model 2 predictions, and `base` is my row of the baseline.
- `can_pick` is True for popular movies I *haven't* rated. There's no point recommending a movie I've already seen.
- Inside `top`, `np.where(can_pick, scores, -np.inf)` gives every other movie a score of −∞, so they can never make the list. The minus sign in `np.argsort(-...)` sorts from highest to lowest, and `[:n]` keeps the top `n`.
- `set(...) & set(...)` keeps only the movies that show up in both lists.


**All 10 movies are in both lists.** Model 2 only changes the order a little. For example, *The Usual Suspects* moves from 6th on the baseline's list to 3rd, and *Star Wars* moves from 8th to 4th. Both lists are basically "movies almost everyone rates highly" (*A Close Shave*, *Schindler's List*, *Casablanca*, and so on). The baseline already knows about those, and the SVD only nudges them around. All the scores are around 3.8 to 4.1 stars because my average is on the low side, and the baseline adds how generous I am to every movie.

### Step 5: what did the personal part change?

In [12]:
personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
  +0.16 Clockwork Orange, A (1971)
  +0.16 Beavis and Butt-head Do America (1996)
  +0.15 GoodFellas (1990)
  +0.15 Trainspotting (1996)
  +0.14 Apocalypse Now (1979)
  +0.13 Clerks (1994)
Moved down most for me:
  -0.25 Contact (1997)
  -0.22 Air Force One (1997)
  -0.12 Conspiracy Theory (1997)
  -0.12 Dante's Peak (1997)
  -0.11 Star Trek: First Contact (1996)
  -0.11 Phenomenon (1996)


`personal = mine - base` is exactly what the SVD added on top of the baseline, just for me. `top(personal, 6)` gives the 6 movies it pushed up the most. `top(-personal, 6)` flips the signs, so it gives the 6 it pushed down the most.


**Moved up for me:** *A Clockwork Orange*, *Beavis and Butt-head Do America*, *GoodFellas*, *Trainspotting*, *Apocalypse Now*, and *Clerks*. These are mostly darker or edgier cult movies. That fits with me giving *Pulp Fiction* 5 stars, and in factor 1, *Pulp Fiction*, *A Clockwork Orange*, and *Trainspotting* were all at the same end.

**Moved down for me:** *Contact*, *Air Force One*, *Conspiracy Theory*, *Dante's Peak*, *Star Trek: First Contact*, and *Phenomenon*. These are all big 1996–97 studio movies, the same kind as *Independence Day* and *Twister*, which I rated low. *Air Force One* and *Phenomenon* were even at one end of factor 2 (Exercise 1).

So from just 14 ratings, the model figured out that I like edgy cult movies and don't like the mid-90s blockbusters, without being told any genres. But the changes are small: the biggest drop is 0.25 stars (*Contact*) and the biggest jump is only 0.16. That's why my top 10 didn't change at all.

**Key idea:** a prediction is *baseline + personal part*. The baseline (how good the movie is and how generous I am as a rater) is the big part. The SVD's personal part is only a correction of a few tenths of a star. That's why Model 2 only beat the baseline by about 3% in Milestone 3, and it's also why its recommendations still make sense.

# Practice Exercises

### Exercise 1 — Name the factors

In [13]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)
print("Biggest 5 singular values:", np.round(s[:5], 1))
show_factor(2)

Biggest 5 singular values: [48.3 41.6 35.3 34.6 31.4]
Factor 2, one end:
  -0.099 Independence Day (ID4) (1996)
  -0.092 Twister (1996)
  -0.077 Eraser (1996)
  -0.074 Air Force One (1997)
  -0.067 Saint, The (1997)
  -0.066 Phenomenon (1996)
Factor 2, other end:
  +0.203 Star Wars (1977)
  +0.199 Pulp Fiction (1994)
  +0.173 Godfather, The (1972)
  +0.168 Empire Strikes Back, The (1980)
  +0.151 Fargo (1996)
  +0.146 Return of the Jedi (1983)


Step 3 replaced `U`, `s`, and `Vt` with the SVD that includes me, so the first line redoes the SVD of `X` from Section 2 (just the 943 real users) before looking at factor 2.

**Factor 2:** one end is a bunch of big 1996–97 movies that were popular but not that loved (*Independence Day*, *Twister*, *Eraser*, *Air Force One*, *The Saint*, *Phenomenon*). The other end is famous all-time favorites (*Star Wars*, *Pulp Fiction*, *The Godfather*, *The Empire Strikes Back*, *Fargo*, *Return of the Jedi*).

My names for the three factors:
- **Factor 1: dark cult movies vs. feel-good blockbusters**
- **Factor 2: all-time classics vs. forgettable new blockbusters**
- **Factor 3: sci-fi and action vs. musicals and family movies**

**Factor 2 was the hardest to name.** Its "classics" end mixes *Star Wars* and *Pulp Fiction*, which were on opposite ends of factor 1, so it isn't a genre. The best I can say is that it's about how much someone loves the famous classics compared to the newer hits from right when this data was collected.

**Why some factors don't match any human idea:** the SVD doesn't know what a genre is. It just picks whatever direction explains the most leftover variation in the numbers, and every factor has to be perpendicular to the ones before it. So one factor can end up mixing a few different ideas together, or picking up stuff like *when* people rated movies instead of what kind of movie it is. Also, the 3rd and 4th singular values are really close (35.3 and 34.6), and when two singular values are almost tied, the SVD can mix those two factors in different ways, so their exact "meaning" isn't very stable.

### Exercise 2 — My own ratings

In [14]:
casablanca = 483 - 1
print(titles[casablanca], "-", n_ratings[casablanca], "ratings")
print(f"baseline {base[casablanca]:.2f} + personal part {personal[casablanca]:+.2f}"
      f" = Model 2 {mine[casablanca]:.2f}")

Casablanca (1942) - 243 ratings
baseline 3.94 + personal part +0.03 = Model 2 3.96


**How many of my top 10 are also in the baseline's top 10?** All 10 of them. Model 2 only changed the order.

**Do the "moved up" and "moved down" lists make sense?** Mostly, yes. Pushing up dark cult movies like *GoodFellas* and *Trainspotting* makes sense for someone who loved *Pulp Fiction*, and pushing down *Air Force One* and *Dante's Peak* makes sense for someone who rated *Twister* and *Independence Day* low. The one that doesn't fit is *Beavis and Butt-head Do America* in the moved-up list. It's a crude cartoon comedy, and nothing I rated is really like it. It probably landed there because people who like edgy cult movies also tended to like it, so it sits on the same "not for everyone" side of factor 1 as *Pulp Fiction*. (It's still nowhere near my top 10, since its baseline is low.)

**One recommendation I think is wrong: *Casablanca*,** number 5 on my list. It's a romance, and I gave *Dirty Dancing*, *Grease*, and *Titanic* only 1 or 2 stars. The cell above shows why it's there anyway: its baseline is 3.94 because almost everyone rates it highly, and the personal part is just +0.03. The SVD didn't push it down at all. The model doesn't know what a "romance" is. It only knows who liked what. The people who love *Casablanca* are mostly classic-movie fans, not the same people who like *Titanic* or *Dirty Dancing*, so my low ratings for newer romances don't tell the model anything about a black-and-white movie from 1942. The baseline does most of the work, and here it just recommends what everyone likes.

### Exercise 3 — The opposite person

In [15]:
def refit_with(user_ratings, k=10):
    """Step 3 again for a different new user or a different k.

    Returns that user's row of Model 2 predictions and their row of the baseline.
    """
    new_row = np.full((1, R.shape[1]), np.nan)
    for movie, stars in user_ratings.items():
        new_row[0, movie - 1] = stars
    R_new = np.vstack([R, new_row])
    known_new = ~np.isnan(R_new)
    B_new = baseline_matrix(R_new)
    X_new = np.where(known_new, R_new - B_new, 0.0)
    U_new, s_new, Vt_new = np.linalg.svd(X_new, full_matrices=False)
    P_new = rank_k(U_new, s_new, Vt_new, k) + B_new
    return P_new[-1], B_new[-1]

def show_moves(personal, n=6):
    """Print the n movies a personal part pushed up most and down most."""
    print("Moved up most:")
    for m in top(personal, n):
        print(f"  {personal[m]:+.2f} {titles[m]}")
    print("Moved down most:")
    for m in top(-personal, n):
        print(f"  {personal[m]:+.2f} {titles[m]}")

To redo Step 3 with different ratings or a different k, I put it in a function. `refit_with` does exactly what Step 3 did: add the new user as row 944, recompute the baseline, take the SVD of the deviations, keep k factors, and return that user's predictions and baseline. It uses new variable names inside (`U_new` and so on), so it doesn't overwrite anything from earlier. `show_moves` is the printing part of Step 5.

In [16]:
friend = {movie: 6 - stars for movie, stars in my_ratings.items()}
print("Friend's ratings:", friend)

friend_mine, friend_base = refit_with(friend)
friend_personal = friend_mine - friend_base
show_moves(friend_personal)

fr_up, fr_down = top(friend_personal, 6), top(-friend_personal, 6)
print("\nUp for me, down for my friend:",
      [titles[m] for m in top(personal, 6) if m in fr_down])
print("Down for me, up for my friend:",
      [titles[m] for m in top(-personal, 6) if m in fr_up])

shift = (friend_base - base)[can_pick]
print("\nSame baseline top 10, same order:", list(top(friend_base)) == list(top(base)))
print("Friend's baseline minus mine, smallest and biggest:",
      round(shift.min(), 4), round(shift.max(), 4))

Friend's ratings: {1: 1, 71: 2, 56: 1, 64: 1, 127: 2, 204: 2, 82: 2, 257: 3, 94: 3, 313: 4, 121: 4, 118: 5, 451: 5, 155: 5}
Moved up most:
  +0.24 Air Force One (1997)
  +0.16 Contact (1997)
  +0.14 Eraser (1996)
  +0.14 Dante's Peak (1997)
  +0.14 Phenomenon (1996)
  +0.13 Conspiracy Theory (1997)
Moved down most:
  -0.30 Liar Liar (1997)
  -0.28 GoodFellas (1990)
  -0.27 Apocalypse Now (1979)
  -0.25 Trainspotting (1996)
  -0.24 Godfather: Part II, The (1974)
  -0.24 Scream (1996)

Up for me, down for my friend: ['GoodFellas (1990)', 'Trainspotting (1996)', 'Apocalypse Now (1979)']
Down for me, up for my friend: ['Contact (1997)', 'Air Force One (1997)', 'Conspiracy Theory (1997)', "Dante's Peak (1997)", 'Phenomenon (1996)']

Same baseline top 10, same order: True
Friend's baseline minus mine, smallest and biggest: -0.2857 -0.2857


I made my friend's ratings with `6 - stars`. That turns every 5 into a 1 and every 1 into a 5 like the exercise says, and it also turns 4s into 2s (3s stay 3s), so the friend really has the opposite taste on every movie.

**Which movies swap?** *GoodFellas*, *Trainspotting*, and *Apocalypse Now* go from moved up for me to moved down for my friend. *Contact*, *Air Force One*, *Conspiracy Theory*, *Dante's Peak*, and *Phenomenon* go from moved down for me to moved up for my friend. So the personal part mostly flips. It isn't a perfect mirror, though: some movies only show up on one list (like *Liar Liar* and *Scream* moving down for my friend), and my friend's changes are bigger than mine. That's because each deviation is measured from what the baseline expected. When I give *Shawshank* 5 stars, that's what everyone gives it, so it's barely a surprise. When my friend gives it 1 star, that's a huge surprise, so the friend's deviations aren't just the opposite of mine.

**Does the baseline top 10 change?** No. It's the same 10 movies in exactly the same order, and every score just drops by the same 0.2857 stars. The baseline is μ + b_user + b_item, and the only part about the user is b_user, a single number for how generous they are. My friend's average is 6 − 3.14 = 2.86, which is about 0.29 lower than mine, so every one of their baseline scores is 0.29 lower, and the order can't change. The baseline knows *how much* someone likes movies in general, but nothing about *which* movies. That's the personal part's job.

### Exercise 4 — Does k matter here?

In [17]:
for k in [2, 10, 50]:
    mine_k, base_k = refit_with(my_ratings, k=k)
    print(f"--- k = {k}")
    show_moves(mine_k - base_k)

--- k = 2
Moved up most:
  +0.14 Fargo (1996)
  +0.11 Leaving Las Vegas (1995)
  +0.11 2001: A Space Odyssey (1968)
  +0.11 Raising Arizona (1987)
  +0.10 Clockwork Orange, A (1971)
  +0.09 Trainspotting (1996)
Moved down most:
  -0.10 Air Force One (1997)
  -0.09 Phenomenon (1996)
  -0.09 Saint, The (1997)
  -0.09 Rock, The (1996)
  -0.08 Mission: Impossible (1996)
  -0.08 Eraser (1996)
--- k = 10
Moved up most:
  +0.16 Clockwork Orange, A (1971)
  +0.16 Beavis and Butt-head Do America (1996)
  +0.15 GoodFellas (1990)
  +0.15 Trainspotting (1996)
  +0.14 Apocalypse Now (1979)
  +0.13 Clerks (1994)
Moved down most:
  -0.25 Contact (1997)
  -0.22 Air Force One (1997)
  -0.12 Conspiracy Theory (1997)
  -0.12 Dante's Peak (1997)
  -0.11 Star Trek: First Contact (1996)
  -0.11 Phenomenon (1996)
--- k = 50
Moved up most:
  +0.34 Dead Man Walking (1995)
  +0.25 Liar Liar (1997)
  +0.23 Seven (Se7en) (1995)
  +0.21 Terminator 2: Judgment Day (1991)
  +0.20 Raiders of the Lost Ark (1981)
  +0.

**k = 2:** moved up is *Fargo*, *Leaving Las Vegas*, *2001: A Space Odyssey*, *Raising Arizona*, *A Clockwork Orange*, and *Trainspotting*, and moved down is *Air Force One*, *Phenomenon*, *The Saint*, *The Rock*, *Mission: Impossible*, and *Eraser*. With only 2 factors, the model can only say one broad thing about me: "likes cult and indie movies, doesn't like big blockbusters." The changes are also small (0.14 stars at most).

**k = 10:** the lists from Step 5. It's the same general idea, but more specific (*GoodFellas*, *Apocalypse Now*, *Clerks*).

**k = 50:** the changes get bigger and weirder. *Dead Man Walking* goes up 0.34, and *Liar Liar* (a Jim Carrey comedy) and *Emma* (a Jane Austen romance) move up too, even though I rated *Titanic* and *Dirty Dancing* low. Meanwhile *This Is Spinal Tap* and *The Fifth Element* move down. It's hard to see any pattern anymore.

**Connection to overfitting:** with 50 factors, the model has so much freedom that it can match my 14 ratings really closely, so it starts fitting little quirks in them instead of my actual taste. That's the same thing we saw in Milestone 3: Model 2's validation RMSE got better up to around k = 10–15 and then got worse (0.9583 at k = 50), because the extra factors were fitting noise. k = 2 has the opposite problem. It's too simple, so it can only see one broad pattern, and it was also worse in Milestone 3 (0.9570). k = 10 is in the sweet spot, which is why its lists make the most sense.

# What we found

Without being told any genres, the model found patterns like "sci-fi and action vs. musicals and family movies" on its own, and from just 14 of my ratings it moved edgy cult movies like *A Clockwork Orange* and *GoodFellas* up and big 1996–97 blockbusters like *Air Force One* down, but only by a quarter of a star at most, so all 10 of its top picks were the same crowd favorites that the simple average recommends.